In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/resources/utils

In [0]:
logger = get_logger()

In [0]:
def dedupe(df: DataFrame, subset: list = None) -> DataFrame:
    """ Drop duplicate rows from a DataFrame."""
    
    return df.dropDuplicates(subset) if subset else df.dropDuplicates()

In [0]:
def build_profit_by_year(enriched_df):
    """Aggregate profit by year, category, sub_category, customer."""
    profit_df = (
        enriched_df
        .withColumn("Year", F.year(F.col("order_date")))
        .groupBy("Year", "category", "sub_category", "customer_name")
        .agg(F.round(F.sum("profit"), 2).alias("total_profit"))
        .withColumnRenamed("category", "product_category")
        .withColumnRenamed("sub_category", "product_sub_category")
        .withColumnRenamed("customer_name", "customer")
    )
    return profit_df

In [0]:
def main_enrich():
    try:

        profit_by_year = build_profit_by_year(
            spark.read.table("az_ci_de_dbs.ecom_dev.enriched_orders")
        )
        profit_by_year = dedupe(profit_by_year)
        
        delta_writer(profit_by_year, "az_ci_de_dbs.ecom_dev.profit_by_year")

        logger.info("Processing completed successfully.")

    except Exception as e:
        logger.error(f"Error during processing: {e}")